# 02 · Componentes de rentabilidad mensual

En el primer cuaderno vimos qué señal propone cada regla. Aquí preparamos la variación posterior de precios, el dividendo aproximado y la alternativa de efectivo. **Todavía no aplicamos las señales ni calculamos resultados de estrategias.**

Antes de empezar, prepara Shiller con el notebook 01 o la opción 1 del menú. Selecciona `.venv` y reinicia el kernel si utilizaste una versión anterior del código. Ejecuta las celdas en orden.

Solo analizaremos 1927–1989. Los resultados sobre promedios mensuales son una aproximación histórica y no representan operaciones ejecutadas a esos precios.

## 1. Comprobar la entrada

Esta celda lee Shiller ya preparado, sin descargarlo ni modificarlo. Si cambiaste su configuración o falta la tabla, vuelve primero al notebook 01. La historia previa permite obtener el precio de diciembre de 1926 para calcular la primera variación de enero de 1927.

In [1]:
from pathlib import Path
from math import isclose

import pandas as pd
from IPython.display import display

from tfm.config import DEFAULT_CONFIG, load_config
from tfm.data import prepare_french_data, prepare_return_components, read_result

CONFIG = DEFAULT_CONFIG
configuracion = load_config(CONFIG)
mensuales, _ = read_result(configuracion, "shiller_monthly")
print(f"Configuración: {CONFIG}")
print(f"Shiller preparado: {len(mensuales)} meses, hasta {mensuales.period.iloc[-1]}.")
print("La evaluación desde 1990 sigue deshabilitada.")

Configuración: /Users/pmb/repos/UOCDataScienceTFM/configs/estudio.toml
Shiller preparado: 769 meses, hasta 1989-12.
La evaluación desde 1990 sigue deshabilitada.


## 2. Preparar la alternativa de efectivo y los componentes

`RF` procede del [CSV mensual de tres factores de Kenneth French](https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/Data_Library/f-f_factors.html). Representa el rendimiento mensual del Treasury bill a un mes; no es el rendimiento del mercado ni del S&P.

El CSV expresa tasas en porcentaje: **0,20 en origen se convierte en 0,002**, equivalente al 0,20 % mensual. Como ya es un dato mensual, no se divide entre doce. Solo se selecciona RF de 1927–1989, los demás factores y los retornos anuales quedan fuera.

**Esta celda escribe archivos.** Reutiliza `data/raw/french_factors.zip` si existe, si falta lo descarga. Para importar un ZIP local, sustituye `None` por `Path("/ruta/al/F-F_Research_Data_Factors_CSV.zip")`. El original completo puede contener datos posteriores, pero esos importes no se incorporan a la tabla preparada.

Shiller debe estar preparado y RF debe cubrir exactamente los mismos meses de desarrollo. Si falta un mes, el proceso se detiene, por tanto no se rellena ni se elimina.

In [2]:
archivo_french_local = None

resumen_rf = prepare_french_data(CONFIG, file=archivo_french_local)
resumen_componentes = prepare_return_components(CONFIG)

display(pd.DataFrame([
    {"resultado": "RF de French", "meses": resumen_rf["rows"], "archivo": resumen_rf["path"]},
    {"resultado": "Componentes", "meses": resumen_componentes["rows"], "archivo": resumen_componentes["path"]},
]))

,resultado,meses,archivo
0,RF de French,756,/Users/pmb/repos/UOCDataScienceTFM/data/proces...
1,Componentes,756,/Users/pmb/repos/UOCDataScienceTFM/data/proces...


## 3. Qué representa cada columna

Para un mes $t$, usamos el promedio $P_t$ y el del mes anterior $P_{t-1}$:

| Columna | Cálculo | Interpretación |
|---|---|---|
| `price_return` | $P_t/P_{t-1}-1$ | Variación entre promedios mensuales |
| `dividend_monthly_approx` | $D_t/12$ | Dividendo aproximado en puntos del índice |
| `dividend_return_approx` | $(D_t/12)/P_{t-1}$ | Componente de rentabilidad por dividendos aproximados |
| `total_return_approx` | Precio + dividendos aproximados | Rentabilidad total aproximada del índice |
| `rf_return` | RF del CSV / 100 | Rentabilidad mensual del efectivo |

Las columnas de rentabilidad se guardan en **decimal**. Por ejemplo, 0,01 significa 1 %. D/12 es una convención contable sobre una serie anualizada e interpolada, no un pago mensual observado ni información necesariamente publicada entonces. RF es un resultado del mes, no una variable para predecir ese mismo resultado.

In [3]:
componentes, informe = read_result(configuracion, "monthly_return_components")
print(f"Periodo: {informe['first_period']} → {informe['last_period']} | {len(componentes)} meses")
display(componentes.head())
display(componentes.tail())

Periodo: 1927-01 → 1989-12 | 756 meses


,asset,period,previous_period,price_previous_month,price_monthly_mean,dividend_annualized,dividend_monthly_approx,price_return,dividend_return_approx,total_return_approx,rf_return
0,SHILLER_SP_COMPOSITE,1927-01,1926-12,13.49,13.40,0.6967,0.058058,-0.006672,0.004304,-0.002368,0.0025
1,SHILLER_SP_COMPOSITE,1927-02,1927-01,13.40,13.66,0.7033,0.058608,0.019403,0.004374,0.023777,0.0026
2,SHILLER_SP_COMPOSITE,1927-03,1927-02,13.66,13.87,0.7100,0.059167,0.015373,0.004331,0.019705,0.0030
3,SHILLER_SP_COMPOSITE,1927-04,1927-03,13.87,14.21,0.7167,0.059725,0.024513,0.004306,0.028819,0.0025
4,SHILLER_SP_COMPOSITE,1927-05,1927-04,14.21,14.70,0.7233,0.060275,0.034483,0.004242,0.038724,0.0030


,asset,period,previous_period,price_previous_month,price_monthly_mean,dividend_annualized,dividend_monthly_approx,price_return,dividend_return_approx,total_return_approx,rf_return
751,SHILLER_SP_COMPOSITE,1989-08,1989-07,331.9,346.6,10.5467,0.878892,0.044290,0.002648,0.046939,0.0074
752,SHILLER_SP_COMPOSITE,1989-09,1989-08,346.6,347.3,10.7300,0.894167,0.002020,0.002580,0.004599,0.0065
753,SHILLER_SP_COMPOSITE,1989-10,1989-09,347.3,347.4,10.7967,0.899725,0.000288,0.002591,0.002879,0.0068
754,SHILLER_SP_COMPOSITE,1989-11,1989-10,347.4,340.2,10.9233,0.910275,-0.020725,0.002620,-0.018105,0.0069
755,SHILLER_SP_COMPOSITE,1989-12,1989-11,340.2,348.6,11.0600,0.921667,0.024691,0.002709,0.027401,0.0061


## 4. Desglosar el resultado de un mes

Elige un mes objetivo del desarrollo. Por defecto usamos mayo de 1975, cuyo mes anterior es abril de 1975. Aquí sí observamos la variación posterior que se podrá asociar a una señal decidida en abril, todavía no le aplicamos la posición de momentum o reversión.

Primero mostramos los importes de entrada, en puntos del índice. Después mostramos los componentes de rentabilidad en porcentaje para facilitar su lectura.

In [4]:
MES_OBJETIVO = "1975-05"
seleccion = componentes.loc[componentes.period.eq(MES_OBJETIVO)]
if seleccion.empty:
    raise ValueError("Elige un mes entre el inicio y el fin del desarrollo preparado.")
caso = seleccion.iloc[0]

print(f"Periodo: {caso.previous_period} → {caso.period}")
display(pd.DataFrame([
    {"dato": "P del mes anterior", "puntos del índice": caso.price_previous_month},
    {"dato": "P del mes objetivo", "puntos del índice": caso.price_monthly_mean},
    {"dato": "D anualizado del mes objetivo", "puntos del índice": caso.dividend_annualized},
    {"dato": "D/12 aproximado", "puntos del índice": caso.dividend_monthly_approx},
]).round(6))

filas = [("Variación de precio", "price_return"),
         ("Dividendo aproximado", "dividend_return_approx"),
         ("Total aproximado del índice", "total_return_approx"),
         ("Alternativa de efectivo (RF)", "rf_return")]
display(pd.DataFrame([{"componente": nombre, "rentabilidad (%)": 100 * caso[columna]}
                      for nombre, columna in filas]).round(6))

assert isclose(caso.price_return, caso.price_monthly_mean / caso.price_previous_month - 1, abs_tol=1e-12)
assert isclose(caso.dividend_return_approx, caso.dividend_annualized / 12 / caso.price_previous_month, abs_tol=1e-12)
assert isclose(caso.total_return_approx, caso.price_return + caso.dividend_return_approx, abs_tol=1e-12)

Periodo: 1975-04 → 1975-05


,dato,puntos del índice
0,P del mes anterior,84.720000
1,P del mes objetivo,90.100000
2,D anualizado del mes objetivo,3.696670
3,D/12 aproximado,0.308056


,componente,rentabilidad (%)
0,Variación de precio,6.350331
1,Dividendo aproximado,0.363616
2,Total aproximado del índice,6.713947
3,Alternativa de efectivo (RF),0.440000


**El total del índice suma precio y dividendo aproximado. RF es la alternativa de efectivo y no se añade a ese total.**

Para pasar a resultados de estrategias tendremos que aplicar la posición: comprado, fuera o vendido. En una posición vendedora se deberán dividendos y se aplicarán los supuestos de préstamo y financiación. Los escenarios de préstamo acordados siguen documentados, pero aquí no se descuenta ninguno.